In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# NOTE: calibrated-explanations>=1.0.0rc2 is not yet published to PyPI, so this editable install will fail until CE 1.0.0rc2 is released. Run this cell only in a dev environment where a matching CE build is already installed (e.g. an editable checkout); otherwise skip it and rely on the environment's already-installed package.
# import subprocess
# import sys
# from pathlib import Path

# package_dir = Path.cwd().resolve()
# if package_dir.name == 'examples':
#     package_dir = package_dir.parent
# else:
#     repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
#     if repo_candidate.exists():
#         package_dir = repo_candidate

# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])


# Local Ensured Plotly

This notebook demonstrates the canonical Plotly ensured style `plotly.local.ensured`. The older style name `plotly.local.ensured_triangular` is deprecated.

Ensured plot semantics remain aligned with CE's native view:

- x-axis = probability in probabilistic mode, prediction in regression mode
- y-axis = uncertainty
- original marker = original prediction
- arrows = predictive movement from the original point to alternative or rule points
- hover reveals rule conditions and interval metadata
- feature controls provide searchable hide/show toggles for feature groups
- side panel is a text detail view that updates when a rule point is clicked
- roles such as counterfactual, counterpotential, semifactual, ensured, and pareto are shown only when available or defensibly inferable
- arrows and alternatives do not imply causal actionability
- `filter_top` keeps dense ensured plots readable

In [3]:
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from ce_visualization_plotly.plugin import register_plotly_visualization_components
from crepes.extras import DifficultyEstimator
from sklearn.datasets import make_classification, make_regression
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split

register_plotly_visualization_components()  # explicit, idempotent registration
np.set_printoptions(precision=3, suppress=True)


## Classification Example

The current CE alternative custom-style routing works at the collection level, so the ensured examples below use `explore_alternatives(...)` and call `.plot(...)` on the returned collection with a single query instance.

When `feature_checklist=True`, the plot renders a searchable feature control panel. When `side_panel=True`, click a blue rule point to populate the text detail panel.

In [4]:
X_cls, y_cls = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=7,
)

x_proper_cls, x_holdout_cls, y_proper_cls, y_holdout_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.40,
    random_state=7,
    stratify=y_cls,
)
x_cal_cls, X_query_cls, y_cal_cls, y_query_cls = train_test_split(
    x_holdout_cls,
    y_holdout_cls,
    test_size=0.50,
    random_state=7,
    stratify=y_holdout_cls,
)

classification_model = LogisticRegression(random_state=7, solver='liblinear')
classification_explainer = WrapCalibratedExplainer(classification_model)
classification_explainer.fit(x_proper_cls, y_proper_cls)
assert classification_explainer.fitted is True
classification_explainer.calibrate(x_cal_cls, y_cal_cls)
assert classification_explainer.calibrated is True
classification_alternatives = classification_explainer.explore_alternatives(X_query_cls[:1])
len(classification_alternatives.explanations)

1

In [5]:
classification_plot = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
)

In [6]:
classification_checklist_plot = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
    feature_checklist=True,
)

C:\Users\loftuw\AppData\Local\Temp\ipykernel_36320\2930542754.py:1: UserWarning: AlternativeExplanations.plot forwarding plot keyword arguments to downstream renderers/plugins: ['feature_checklist']. If this is a typo of a built-in argument (['bins', 'class_idx', 'instance_index', 'low_high_percentiles', 'renderer', 'return_plot_spec', 'style']), it will be silently ignored by the renderer.
  classification_checklist_plot = classification_alternatives.plot(


In [7]:
classification_panel_plot = classification_alternatives.add_conjunctions(max_rule_size=7).plot(
    style='plotly.local.ensured',
    show=True,
    side_panel=True,
)

C:\Users\loftuw\AppData\Local\Temp\ipykernel_36320\3840242858.py:1: UserWarning: AlternativeExplanations.plot forwarding plot keyword arguments to downstream renderers/plugins: ['side_panel']. If this is a typo of a built-in argument (['bins', 'class_idx', 'instance_index', 'low_high_percentiles', 'renderer', 'return_plot_spec', 'style']), it will be silently ignored by the renderer.
  classification_panel_plot = classification_alternatives.add_conjunctions(max_rule_size=7).plot(


In [8]:
classification_full_plot = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    feature_checklist=True,
    side_panel=True,
)

C:\Users\loftuw\AppData\Local\Temp\ipykernel_36320\3839638228.py:1: UserWarning: AlternativeExplanations.plot forwarding plot keyword arguments to downstream renderers/plugins: ['feature_checklist', 'side_panel']. If this is a typo of a built-in argument (['bins', 'class_idx', 'instance_index', 'low_high_percentiles', 'renderer', 'return_plot_spec', 'style']), it will be silently ignored by the renderer.
  classification_full_plot = classification_alternatives.plot(


In [9]:
classification_export = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=False,
    filename='ensured_classification.html',
    filter_top=20,
    feature_checklist=True,
    side_panel=True,
)
classification_export.saved_paths

C:\Users\loftuw\AppData\Local\Temp\ipykernel_36320\3331741489.py:1: UserWarning: AlternativeExplanations.plot forwarding plot keyword arguments to downstream renderers/plugins: ['feature_checklist', 'side_panel']. If this is a typo of a built-in argument (['bins', 'class_idx', 'instance_index', 'low_high_percentiles', 'renderer', 'return_plot_spec', 'style']), it will be silently ignored by the renderer.
  classification_export = classification_alternatives.plot(


('ensured_classification.html',)

## Regression Example

Regression ensured plots use the same Plotly style, but the x-axis represents the calibrated prediction value instead of a class probability.

In [10]:
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=8,
    n_informative=5,
    noise=0.2,
    random_state=11,
)

x_proper_reg, x_holdout_reg, y_proper_reg, y_holdout_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.40,
    random_state=11,
)
x_cal_reg, X_query_reg, y_cal_reg, y_query_reg = train_test_split(
    x_holdout_reg,
    y_holdout_reg,
    test_size=0.50,
    random_state=11,
)

regression_model = LinearRegression()
regression_explainer = WrapCalibratedExplainer(regression_model)
regression_explainer.fit(x_proper_reg, y_proper_reg)
assert regression_explainer.fitted is True
regression_explainer.calibrate(x_cal_reg, y_cal_reg)
assert regression_explainer.calibrated is True

regression_explainer.set_difficulty_estimator(
    DifficultyEstimator().fit(X=x_proper_reg, y=y_proper_reg, scaler=True)
)

regression_alternatives = regression_explainer.explore_alternatives(
    X_query_reg[:1],
    low_high_percentiles=(5, 95),
)
len(regression_alternatives.explanations)

1

In [11]:
regression_plot = regression_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    feature_checklist=True,
    side_panel=True,
)

C:\Users\loftuw\AppData\Local\Temp\ipykernel_36320\2435494078.py:1: UserWarning: AlternativeExplanations.plot forwarding plot keyword arguments to downstream renderers/plugins: ['feature_checklist', 'side_panel']. If this is a typo of a built-in argument (['bins', 'class_idx', 'instance_index', 'low_high_percentiles', 'renderer', 'return_plot_spec', 'style']), it will be silently ignored by the renderer.
  regression_plot = regression_alternatives.plot(


In [12]:
regression_full_plot = regression_alternatives.add_conjunctions(max_rule_size=7).plot(
    style='plotly.local.ensured',
    show=True,
    feature_checklist=True,
    side_panel=True,
)

C:\Users\loftuw\AppData\Local\Temp\ipykernel_36320\2697711323.py:1: UserWarning: AlternativeExplanations.plot forwarding plot keyword arguments to downstream renderers/plugins: ['feature_checklist', 'side_panel']. If this is a typo of a built-in argument (['bins', 'class_idx', 'instance_index', 'low_high_percentiles', 'renderer', 'return_plot_spec', 'style']), it will be silently ignored by the renderer.
  regression_full_plot = regression_alternatives.add_conjunctions(max_rule_size=7).plot(


In [13]:
regression_export = regression_alternatives.plot(
    style='plotly.local.ensured',
    show=False,
    filename='ensured_regression.html',
    filter_top=20,
    feature_checklist=True,
    side_panel=True,
)
regression_export.saved_paths

C:\Users\loftuw\AppData\Local\Temp\ipykernel_36320\1014588487.py:1: UserWarning: AlternativeExplanations.plot forwarding plot keyword arguments to downstream renderers/plugins: ['feature_checklist', 'side_panel']. If this is a typo of a built-in argument (['bins', 'class_idx', 'instance_index', 'low_high_percentiles', 'renderer', 'return_plot_spec', 'style']), it will be silently ignored by the renderer.
  regression_export = regression_alternatives.plot(


('ensured_regression.html',)